# Feature generation

Builds the feature tables `Datasets/<market>_complexity_datasets/<ID>_MASTER.csv`
used by `EAAI_experiments/`: permutation entropy H, statistical complexity C and
Fisher information F (ordpy; D = 3, tau = 1) and conventional rolling statistics
(BASE) over backward windows of 50, 75, 100 and 250 observations, on the price,
return and absolute-return channels.

Run with the repository root as the working directory. The first cell defines
the functions; the last two cells set the markets and build the tables. With
`USE_LOCAL_DATA = True` the prices are read from `data/` (the FRED values used in
the study); with `False` the current FRED vintage is downloaded.

In [ ]:
import math
import warnings
import json

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import ordpy

try:
    # needed only to download the current FRED vintage (USE_LOCAL_DATA = False)
    from pandas_datareader import data as web
except ImportError:
    web = None


# ============================================================
# 1. CONFIGURATION
# ============================================================

@dataclass
class DatasetConfig:
    """
    Configuration for feature-set generation.

    No targets, forward returns, or lagged complexity features
    are generated here.
    """

    # --------------------------------------------------------
    # FRED
    # --------------------------------------------------------

    series_id: str = "DCOILBRENTEU"

    start: str = "1997-01-07"

    end: Optional[str] = None

    # --------------------------------------------------------
    # Return transformation
    # --------------------------------------------------------
    #
    # "log":
    #     r_t = log(P_t) - log(P_{t-1})
    #
    # "simple":
    #     r_t = P_t / P_{t-1} - 1
    #
    # "diff":
    #     r_t = P_t - P_{t-1}
    #

    return_method: str = "log"

    # --------------------------------------------------------
    # Complexity windows
    # --------------------------------------------------------
    #
    # Example:
    #
    # (125, 250, 500)
    #
    # produces:
    #
    # H_price_w125
    # C_price_w125
    # F_price_w125
    #
    # H_price_w250
    # C_price_w250
    # F_price_w250
    #
    # etc.
    #

    complexity_windows: Tuple[int, ...] = (
        125,
        250,
        500,
    )

    # --------------------------------------------------------
    # Bandt-Pompe parameters
    # --------------------------------------------------------

    embedding_dim: int = 4

    embedding_delay: int = 1

    # --------------------------------------------------------
    # Tie handling
    # --------------------------------------------------------

    tie_precision: Optional[int] = None

    # --------------------------------------------------------
    # BASE features
    # --------------------------------------------------------

    base_rolling_windows: Tuple[int, ...] = (
        5,
        20,
        60,
    )

    # --------------------------------------------------------
    # Dataset alignment
    # --------------------------------------------------------

    align_feature_sets: bool = True


# ============================================================
# 2. CONVENIENT FRED SERIES
# ============================================================

FRED_SERIES = {

    "WTI":
        "DCOILWTICO",

    "BRENT":
        "DCOILBRENTEU",

    "NATURAL_GAS":
        "DHHNGSP",

    "GASOLINE_GULF":
        "DGASUSGULF",
}


# ============================================================
# 3. DOWNLOAD FRED SERIES
# ============================================================

def download_fred_series(
    series_id: str,
    start: str,
    end: Optional[str] = None,
) -> pd.Series:
    """
    Download one FRED series.

    Missing observations are removed rather than forward-filled.
    """

    print(
        f"Downloading FRED series: {series_id}"
    )

    if web is None:
        raise ImportError("pandas_datareader is needed to download from FRED")

    df = web.DataReader(
        series_id,
        "fred",
        start=start,
        end=end,
    )

    series = (
        df.iloc[:, 0]
        .copy()
    )

    series.index = pd.to_datetime(
        series.index
    )

    series = pd.to_numeric(
        series,
        errors="coerce",
    )

    series = (
        series
        .dropna()
        .sort_index()
    )

    series = series[
        ~series.index.duplicated(
            keep="last"
        )
    ]

    series.name = "price"

    print(
        f"Downloaded {len(series):,} observations.\n"
        f"Start: {series.index.min().date()}\n"
        f"End:   {series.index.max().date()}"
    )

    return series


# ============================================================
# 4. RETURN CALCULATION
# ============================================================

def calculate_returns(
    price: pd.Series,
    method: str = "log",
) -> pd.Series:
    """
    Calculate historical/contemporaneous returns.

    No forward-looking values are constructed.
    """

    price = (
        price
        .astype(float)
        .copy()
    )

    # --------------------------------------------------------
    # Log returns
    # --------------------------------------------------------

    if method == "log":

        if (
            price <= 0
        ).any():

            problem_dates = (
                price.index[
                    price <= 0
                ]
            )

            raise ValueError(
                "\nLog returns require strictly positive prices.\n"
                f"Found {len(problem_dates)} non-positive "
                "observations.\n"
                f"First problematic date: "
                f"{problem_dates[0].date()}\n\n"
                "Possible alternatives:\n"
                "1. Use Brent for log-return analysis;\n"
                "2. Set return_method='diff';\n"
                "3. Introduce a dedicated transformation for "
                "non-positive observations."
            )

        returns = (
            np.log(price)
            .diff()
        )

    # --------------------------------------------------------
    # Simple returns
    # --------------------------------------------------------

    elif method == "simple":

        returns = (
            price.pct_change(
                fill_method=None
            )
        )

    # --------------------------------------------------------
    # Price differences
    # --------------------------------------------------------

    elif method == "diff":

        returns = (
            price.diff()
        )

    else:

        raise ValueError(
            "return_method must be one of "
            "'log', 'simple', or 'diff'."
        )

    returns.name = "return"

    return returns


# ============================================================
# 5. VALIDATE COMPLEXITY PARAMETERS
# ============================================================

def validate_complexity_config(
    config: DatasetConfig,
):
    """
    Validate complexity-window and embedding settings.
    """

    if config.embedding_dim < 2:

        raise ValueError(
            "embedding_dim must be >= 2."
        )

    if config.embedding_delay < 1:

        raise ValueError(
            "embedding_delay must be >= 1."
        )

    if len(
        config.complexity_windows
    ) == 0:

        raise ValueError(
            "At least one complexity window is required."
        )

    if any(
        window <= 0
        for window
        in config.complexity_windows
    ):

        raise ValueError(
            "All complexity windows must be positive."
        )

    # Minimum embedding length

    min_embedding_length = (
        1
        +
        (
            config.embedding_dim
            - 1
        )
        *
        config.embedding_delay
    )

    for window in (
        config.complexity_windows
    ):

        if window < min_embedding_length:

            raise ValueError(
                f"Window {window} is too short for "
                f"embedding_dim={config.embedding_dim} and "
                f"embedding_delay={config.embedding_delay}."
            )

    # --------------------------------------------------------
    # Heuristic warning:
    #
    # approximately n >= 5 × D!
    # --------------------------------------------------------

    recommended_minimum = (
        5
        *
        math.factorial(
            config.embedding_dim
        )
    )

    for window in (
        config.complexity_windows
    ):

        if (
            window
            < recommended_minimum
        ):

            warnings.warn(
                f"Complexity window {window} is smaller than "
                f"5 × D! = {recommended_minimum}. "
                "The ordinal probability distribution may be sparse."
            )


# ============================================================
# 6. H, C, F USING ORDPY
# ============================================================

def calculate_ordpy_hcf(
    x: np.ndarray,
    embedding_dim: int,
    embedding_delay: int,
    tie_precision: Optional[int] = None,
):
    """
    Calculate:

        H = normalized permutation entropy
        C = statistical complexity
        F = permutation Fisher information

    using ordpy.
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    if not np.isfinite(
        x
    ).all():

        return (
            np.nan,
            np.nan,
            np.nan,
        )

    # --------------------------------------------------------
    # Complete ordered ordinal probability distribution
    # --------------------------------------------------------

    _, probabilities = (
        ordpy.ordinal_distribution(

            x,

            dx=
                embedding_dim,

            dy=
                1,

            taux=
                embedding_delay,

            tauy=
                1,

            return_missing=
                True,

            ordered=
                True,

            tie_precision=
                tie_precision,
        )
    )

    # --------------------------------------------------------
    # Entropy + statistical complexity
    # --------------------------------------------------------

    H, C = (
        ordpy.complexity_entropy(

            probabilities,

            dx=
                embedding_dim,

            dy=
                1,

            probs=
                True,
        )
    )

    # --------------------------------------------------------
    # Fisher information
    # --------------------------------------------------------

    with warnings.catch_warnings():

        warnings.filterwarnings(
            "ignore",
            message=(
                "Be mindful the correct calculation "
                "of Fisher information*"
            ),
        )

        _, F = (
            ordpy.fisher_shannon(

                probabilities,

                dx=
                    embedding_dim,

                dy=
                    1,

                probs=
                    True,
            )
        )

    return (
        float(H),
        float(C),
        float(F),
    )


# ============================================================
# 7. ROLLING COMPLEXITY:
#    ONE CHANNEL × ONE WINDOW
# ============================================================

def rolling_ordpy_complexity(
    series: pd.Series,
    window: int,
    embedding_dim: int,
    embedding_delay: int,
    prefix: str,
    tie_precision: Optional[int] = None,
) -> pd.DataFrame:
    """
    Calculate rolling H, C and F.

    At time t, only:

        [t-window+1, ..., t]

    is used.
    """

    values = series.to_numpy(
        dtype=float
    )

    n = len(series)

    H_values = np.full(
        n,
        np.nan,
    )

    C_values = np.full(
        n,
        np.nan,
    )

    F_values = np.full(
        n,
        np.nan,
    )

    for end_index in range(
        window - 1,
        n,
    ):

        start_index = (
            end_index
            - window
            + 1
        )

        x = values[
            start_index:
            end_index + 1
        ]

        if not np.isfinite(
            x
        ).all():

            continue

        H, C, F = (
            calculate_ordpy_hcf(

                x=x,

                embedding_dim=
                    embedding_dim,

                embedding_delay=
                    embedding_delay,

                tie_precision=
                    tie_precision,
            )
        )

        H_values[
            end_index
        ] = H

        C_values[
            end_index
        ] = C

        F_values[
            end_index
        ] = F

    result = pd.DataFrame(

        {

            f"H_{prefix}_w{window}":
                H_values,

            f"C_{prefix}_w{window}":
                C_values,

            f"F_{prefix}_w{window}":
                F_values,
        },

        index=
            series.index,
    )

    return result


# ============================================================
# 8. MULTI-WINDOW COMPLEXITY FEATURES
# ============================================================

def create_complexity_features(
    price: pd.Series,
    returns: pd.Series,
    config: DatasetConfig,
) -> pd.DataFrame:
    """
    Calculate H, C and F for:

        price
        returns
        absolute returns

    for each window specified in complexity_windows.

    No lagged complexity features are created.
    """

    validate_complexity_config(
        config
    )

    channels = {

        "price":
            price,

        "return":
            returns,

        "abs_return":
            returns.abs(),
    }

    frames = []

    windows = tuple(
        sorted(
            set(
                config.complexity_windows
            )
        )
    )

    for (
        channel_name,
        channel_series,
    ) in channels.items():

        print(
            "\n"
            f"Complexity channel: {channel_name}"
        )

        for window in windows:

            print(
                f"  Window = {window}"
            )

            frame = (
                rolling_ordpy_complexity(

                    series=
                        channel_series,

                    window=
                        window,

                    embedding_dim=
                        config.embedding_dim,

                    embedding_delay=
                        config.embedding_delay,

                    prefix=
                        channel_name,

                    tie_precision=
                        config.tie_precision,
                )
            )

            frames.append(
                frame
            )

    complexity_features = (
        pd.concat(
            frames,
            axis=1,
        )
    )

    return complexity_features


# ============================================================
# 9. BASE FEATURES
# ============================================================

def create_base_features(
    returns: pd.Series,
    abs_returns: pd.Series,
    config: DatasetConfig,
) -> pd.DataFrame:

    features = pd.DataFrame(
        index=returns.index
    )

    # ========================================================
    # CURRENT MARKET STATE
    # ========================================================

    features["return"] = returns

    features["abs_return"] = abs_returns

    features["squared_return"] = (
        returns ** 2
    )

    # ========================================================
    # ROLLING STATISTICS
    # ========================================================

    for window in config.base_rolling_windows:

        rolling_returns = (
            returns.rolling(
                window=window,
                min_periods=window,
            )
        )

        rolling_abs_returns = (
            abs_returns.rolling(
                window=window,
                min_periods=window,
            )
        )

        # Location
        features[
            f"return_mean_w{window}"
        ] = rolling_returns.mean()

        features[
            f"return_median_w{window}"
        ] = rolling_returns.median()

        # Volatility
        features[
            f"return_std_w{window}"
        ] = rolling_returns.std(
            ddof=1
        )

        # Distribution shape
        features[
            f"return_skew_w{window}"
        ] = rolling_returns.skew()

        features[
            f"return_kurt_w{window}"
        ] = rolling_returns.kurt()

        # Absolute-return dynamics
        features[
            f"abs_return_mean_w{window}"
        ] = rolling_abs_returns.mean()

        features[
            f"abs_return_std_w{window}"
        ] = rolling_abs_returns.std(
            ddof=1
        )

        # Realized volatility
        features[
            f"realized_vol_w{window}"
        ] = (
            returns
            .pow(2)
            .rolling(
                window=window,
                min_periods=window,
            )
            .sum()
            .pow(0.5)
        )

        # Cumulative return
        features[
            f"cumulative_return_w{window}"
        ] = (
            rolling_returns.sum()
        )

    return features


# ============================================================
# 10. BUILD FEATURE SETS
# ============================================================

def build_feature_sets_from_series(
    price: pd.Series,
    config: DatasetConfig,
):
    """
    Build:

        BASE
        PE
        CE
        CEF
        BASE+CEF

    No targets.
    No forward returns.
    No lagged complexity features.

    The price column is retained inside each dataset but is
    NOT included in feature_columns.
    """

    # ========================================================
    # Clean price
    # ========================================================

    price = (
        price
        .copy()
    )

    price.index = pd.to_datetime(
        price.index
    )

    price = pd.to_numeric(
        price,
        errors="coerce",
    )

    price = (
        price
        .dropna()
        .sort_index()
    )

    price = price[
        ~price.index.duplicated(
            keep="last"
        )
    ]

    price.name = "price"

    # ========================================================
    # Returns
    # ========================================================

    returns = (
        calculate_returns(

            price=
                price,

            method=
                config.return_method,
        )
    )

    abs_returns = (
        returns.abs()
    )

    # ========================================================
    # BASE features
    # ========================================================

    print(
        "\nCreating BASE features..."
    )

    base_features = (
        create_base_features(

            returns=
                returns,

            abs_returns=
                abs_returns,

            config=
                config,
        )
    )

    # ========================================================
    # Complexity features
    # ========================================================

    print(
        "\nCreating multi-window complexity features..."
    )

    complexity_features = (
        create_complexity_features(

            price=
                price,

            returns=
                returns,

            config=
                config,
        )
    )

    # ========================================================
    # Feature groups
    # ========================================================

    PE_columns = [

        column

        for column
        in complexity_features.columns

        if column.startswith(
            "H_"
        )
    ]


    CE_columns = [

        column

        for column
        in complexity_features.columns

        if column.startswith(
            (
                "H_",
                "C_",
            )
        )
    ]


    CEF_columns = [

        column

        for column
        in complexity_features.columns

        if column.startswith(
            (
                "H_",
                "C_",
                "F_",
            )
        )
    ]


    BASE_columns = list(
        base_features.columns
    )


    BASE_CEF_columns = (
        BASE_columns
        +
        CEF_columns
    )

    # ========================================================
    # MASTER DATASET
    # ========================================================

    master = pd.concat(
        [
            price.rename("price"),
            base_features,
            complexity_features,
        ],
        axis=1,
    )

    # ========================================================
    # ALIGN DATASETS
    # ========================================================

    if config.align_feature_sets:

        required_columns = (

            ["price"]

            + BASE_columns

            + CEF_columns
        )

        common_index = (

            master[
                required_columns
            ]

            .dropna()

            .index
        )

    else:

        common_index = (
            master.index
        )

    # ========================================================
    # DATASET FACTORY
    # ========================================================

    def make_dataset(
        feature_columns,
    ):
        """
        Create a feature-set dataset.

        'price' is retained in the dataset for later target
        construction, but is NOT considered an ML feature.
        """

        columns = (
            ["price"]
            + feature_columns
        )

        # Remove accidental duplicates while preserving order
        columns = list(
            dict.fromkeys(columns)
        )

        dataset = (
            master.loc[
                common_index,
                columns,
            ]
            .copy()
            .dropna()
        )

        return dataset

    # ========================================================
    # FIVE DATASETS
    # ========================================================

    datasets = {

        "BASE":
            make_dataset(
                BASE_columns
            ),

        "PE":
            make_dataset(
                PE_columns
            ),

        "CE":
            make_dataset(
                CE_columns
            ),

        "CEF":
            make_dataset(
                CEF_columns
            ),

        "BASE+CEF":
            make_dataset(
                BASE_CEF_columns
            ),
    }

    # ========================================================
    # FEATURE COLUMN DICTIONARY
    # ========================================================
    #
    # Important:
    #
    # price is deliberately NOT included here.
    #
    # Therefore:
    #
    # X = df[feature_columns["CEF"]]
    #
    # will automatically exclude price.
    #

    feature_columns = {

        "BASE":
            BASE_columns,

        "PE":
            PE_columns,

        "CE":
            CE_columns,

        "CEF":
            CEF_columns,

        "BASE+CEF":
            BASE_CEF_columns,
    }

    # ========================================================
    # SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 65
    )

    print(
        "FEATURE GENERATION FINISHED"
    )

    print(
        "=" * 65
    )

    for (
        dataset_name,
        dataset,
    ) in datasets.items():

        n_model_features = len(
            feature_columns[
                dataset_name
            ]
        )

        print(
            f"{dataset_name:10s}: "
            f"{dataset.shape[0]:,} observations | "
            f"{n_model_features} model features | "
            f"{dataset.shape[1]} stored columns"
        )

    print(
        "\nNo targets, forward returns, or "
        "lagged complexity features were generated."
    )

    return (
        datasets,
        master,
        feature_columns,
    )


# ============================================================
# 11. FRED WRAPPER
# ============================================================

def build_feature_sets_from_fred(
    config: DatasetConfig,
):
    """
    Download selected FRED commodity and construct the
    five feature sets.
    """

    price = (
        download_fred_series(

            series_id=
                config.series_id,

            start=
                config.start,

            end=
                config.end,
        )
    )

    return (
        build_feature_sets_from_series(

            price=
                price,

            config=
                config,
        )
    )


# ============================================================
# 12. SAVE FEATURE SETS
# ============================================================

def save_feature_sets(
    datasets: Dict[
        str,
        pd.DataFrame,
    ],
    series_id: str,
    folder: str = "complexity_datasets",
):
    """
    Save the five feature sets to CSV.
    """

    output_folder = Path(
        folder
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    for (
        name,
        dataset,
    ) in datasets.items():

        safe_name = (
            name.replace(
                "+",
                "_PLUS_",
            )
        )

        filename = (
            f"{series_id}_{safe_name}.csv"
        )

        path = (
            output_folder
            / filename
        )

        dataset.to_csv(
            path
        )

        print(
            f"Saved: {path}"
        )


# ============================================================
# 13. SAVE MASTER DATASET
# ============================================================

def save_master_dataset(
    master: pd.DataFrame,
    series_id: str,
    folder: str = "complexity_datasets",
):
    """
    Save the complete master feature table.
    """

    output_folder = Path(
        folder
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    path = (
        output_folder
        /
        f"{series_id}_MASTER.csv"
    )

    master.to_csv(
        path
    )

    print(
        f"Saved: {path}"
    )


# ============================================================
# 14. SAVE CONFIGURATION
# ============================================================

def save_dataset_config(
    config,
    folder="complexity_datasets",
):
    """
    Save dataset-generation parameters as JSON.
    """

    output_folder = Path(
        folder
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    path = (
        output_folder
        /
        f"{config.series_id}_CONFIG.json"
    )

    config_dict = (
        asdict(config)
    )

    for (
        key,
        value,
    ) in config_dict.items():

        if isinstance(
            value,
            tuple,
        ):

            config_dict[
                key
            ] = list(
                value
            )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            config_dict,
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(
        f"Saved: {path}"
    )

In [ ]:
# Markets and sample periods of the study (as in Datasets/*/<ID>_CONFIG.json)
MARKET_SERIES = {
    "WTI": ("DCOILWTICO", "1986-01-02"),
    "Brent": ("DCOILBRENTEU", "1987-05-20"),
    "Gasoline": ("DGASUSGULF", "1986-06-02"),
    "Henry_Hub": ("DHHNGSP", "1997-01-07"),
}
END = "2026-08-18"

USE_LOCAL_DATA = True  # False: download the current FRED vintage instead
DATA_DIR = Path("data")
OUT_DIR = Path("Datasets")


def make_config(series_id, start):
    return DatasetConfig(
        series_id=series_id,
        start=start,
        end=END,
        return_method="simple",
        complexity_windows=(50, 75, 100, 250),
        embedding_dim=3,
        embedding_delay=1,
        tie_precision=None,
        base_rolling_windows=(50, 75, 100, 250),
        align_feature_sets=True,
    )


def load_local_price(series_id, start, end):
    """Frozen FRED values; dates without a quotation are already removed."""
    df = pd.read_csv(DATA_DIR / f"{series_id}.csv", index_col=0, parse_dates=True)
    return df["price"].loc[start:end]

In [ ]:
for key, (series_id, start) in MARKET_SERIES.items():
    config = make_config(series_id, start)
    if USE_LOCAL_DATA:
        price = load_local_price(series_id, config.start, config.end)
        datasets, master, feature_columns = build_feature_sets_from_series(
            price=price, config=config
        )
    else:
        datasets, master, feature_columns = build_feature_sets_from_fred(config)
    folder = OUT_DIR / f"{key}_complexity_datasets"
    save_feature_sets(datasets=datasets, series_id=series_id, folder=folder)
    save_master_dataset(master=master, series_id=series_id, folder=folder)
    save_dataset_config(config=config, folder=folder)